# NB03 — Feature Engineering

Membuat fitur dan target forecasting **per sensor** dari data per jam hasil NB00.

**Input:** `data/processed/hourly_per_sensor.csv` (hasil NB00)
**Output:** `data/processed/model_dataset.csv` dan `data/processed/model_config.json`

Setiap cell diberi kode `[NB03-Cxx]` dan diakhiri dengan `✓ [NB03-Cxx] selesai`.

### Aturan waktu (penting)

| Istilah | Arti |
|---|---|
| **Baris jam** `hour_wita = 01:00` | rata-rata data **01:00–01:59 WITA** |
| **Origin** `origin_time = 02:00` | waktu forecast dibuat; data terakhir yang dipakai adalah baris jam 01:00 |
| **Titik grafik** jam 05:00 | rata-rata **3 jam terakhir** = baris jam 02:00, 03:00, 04:00 |
| **Target +3 jam** dari origin 02:00 | nilai titik grafik jam 05:00 |
| **Target +24 jam** dari origin 02:00 | nilai titik grafik jam 02:00 hari berikutnya |

Fitur hanya memakai data **sampai origin**. Target selalu berada **setelah origin**.

> **Asumsi zona waktu:** WITA = jam database − 8 jam (variabel `DB_TO_WITA_HOURS` di NB03-C01).

In [ ]:
# ============================================================
# [NB03-C01] SETUP & KONFIGURASI
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROCESSED_DIR = Path("../data/processed")
INPUT_PATH = PROCESSED_DIR / "hourly_per_sensor.csv"
OUTPUT_PATH = PROCESSED_DIR / "model_dataset.csv"
CONFIG_PATH = PROCESSED_DIR / "model_config.json"

DEVICE_COL = "ispu_device_id"

# ASUMSI ZONA WAKTU
# Dari pola matahari, jam database 8 jam lebih cepat dari WITA.
# Ubah angka ini jika hasil pengecekan database menunjukkan selisih lain.
DB_TO_WITA_HOURS = -8

# Data cuaca baru tersedia mulai 24 April 2026,
# sehingga data dipakai mulai hari penuh pertama setelahnya (WITA).
START_WITA = pd.Timestamp("2026-04-25 00:00")

POLLUTANT_COLS = ["PM25", "PM10", "CO", "O3", "SO2", "NO2"]
WEATHER_COLS = ["TMP", "HUM", "RAIN", "WS", "WD_sin", "WD_cos"]

# Horizon utama (grafik 24 jam, titik setiap 3 jam) + horizon eksperimen
HORIZONS = [3, 6, 9, 12, 15, 18, 21, 24]
EXPERIMENT_HORIZONS = [48, 72]
ALL_HORIZONS = HORIZONS + EXPERIMENT_HORIZONS

# Lag polutan: lag1 = jam terakhir sebelum origin, lag24 = 24 jam sebelum origin
POLLUTANT_LAGS = [1, 2, 3, 6, 12, 24]

# Rata-rata bergerak polutan (jam terakhir)
POLLUTANT_ROLLING = [3, 6, 24]

# Profil 24 jam terakhir: titik grafik 3, 6, ..., 21 jam sebelum origin
PROFILE_HOURS_AGO = [3, 6, 9, 12, 15, 18, 21]

# Titik grafik = rata-rata 3 jam terakhir, minimal 2 dari 3 jam tersedia
BLOCK_HOURS = 3
MIN_HOURS_IN_BLOCK = 2

# Jam titik grafik (WITA)
GRID_HOURS_WITA = [2, 5, 8, 11, 14, 17, 20, 23]

# Proporsi split berdasarkan waktu (sama untuk semua sensor)
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

assert INPUT_PATH.exists(), f"Jalankan NB00 dulu. File tidak ditemukan: {INPUT_PATH}"

print("Input               :", INPUT_PATH.resolve())
print("Konversi zona waktu : WITA = jam database", f"{DB_TO_WITA_HOURS:+d} jam")
print("Mulai data (WITA)   :", START_WITA)
print("Horizon utama       :", HORIZONS)
print("Horizon eksperimen  :", EXPERIMENT_HORIZONS)

print("\n✓ [NB03-C01] selesai")

Input               : D:\Kuliah\KP\forecast-ispu\data\processed\hourly_per_sensor.csv
Konversi zona waktu : WITA = jam database -8 jam
Mulai data (WITA)   : 2026-04-25 00:00:00
Horizon utama       : [3, 6, 9, 12, 15, 18, 21, 24]
Horizon eksperimen  : [48, 72]

✓ [NB03-C01] selesai


In [ ]:
# ============================================================
# [NB03-C02] LOAD DATA NB00 & KONVERSI KE WITA
# ============================================================

hourly = pd.read_csv(INPUT_PATH, parse_dates=["datetime_db"])

hourly["hour_wita"] = hourly["datetime_db"] + pd.Timedelta(hours=DB_TO_WITA_HOURS)

print("Shape :", hourly.shape)
print("Periode (jam database):", hourly["datetime_db"].min(), "→", hourly["datetime_db"].max())
print("Periode (WITA)        :", hourly["hour_wita"].min(), "→", hourly["hour_wita"].max())

# Sanity check konversi: setelah dikonversi, matahari harus paling terik
# sekitar tengah hari WITA (lokasi sensor di Kalimantan Timur).
daily_pattern = (
    hourly.groupby(hourly["hour_wita"].dt.hour)[["UV", "TMP"]]
    .mean()
    .round(2)
)
daily_pattern.index.name = "jam_WITA"

uv_peak_hour = daily_pattern["UV"].idxmax()
tmp_peak_hour = daily_pattern["TMP"].idxmax()

print("\nPola harian setelah konversi:")
print(daily_pattern.T.to_string())
print(f"\nUV tertinggi pada jam   : {uv_peak_hour:02d}:00 WITA")
print(f"Suhu tertinggi pada jam : {tmp_peak_hour:02d}:00 WITA")

assert 11 <= uv_peak_hour <= 13, "Puncak UV tidak di tengah hari. Periksa DB_TO_WITA_HOURS."

print("\n✓ [NB03-C02] selesai")

Shape : (28295, 24)
Periode (jam database): 2026-01-12 17:00:00 → 2026-09-21 17:00:00
Periode (WITA)        : 2026-01-12 09:00:00 → 2026-09-21 09:00:00

Pola harian setelah konversi:
jam_WITA     0      1      2      3      4      5      6      7      8      9      10     11     12     13     14     15     16     17     18     19     20     21     22     23
UV         0.01   0.01   0.01   0.01   0.01   0.01   0.13   0.82   2.43   3.48   4.44   4.93   5.49   5.28   4.32   3.05   1.53   0.53   0.06   0.01   0.01   0.01   0.01   0.01
TMP       26.92  26.71  26.51  26.32  26.16  26.02  25.97  27.34  30.00  32.12  33.37  34.33  34.81  34.90  34.41  33.79  32.59  31.26  29.72  28.70  28.14  27.73  27.40  27.16

UV tertinggi pada jam   : 12:00 WITA
Suhu tertinggi pada jam : 13:00 WITA

✓ [NB03-C02] selesai


In [ ]:
# ============================================================
# [NB03-C03] FILTER PERIODE & KOLOM
# ============================================================

df = hourly[hourly["hour_wita"] >= START_WITA].copy()

df = df[[DEVICE_COL, "hour_wita"] + POLLUTANT_COLS + WEATHER_COLS]
df = df.sort_values([DEVICE_COL, "hour_wita"]).reset_index(drop=True)

summary = df.groupby(DEVICE_COL).agg(
    jumlah_jam=("hour_wita", "count"),
    mulai=("hour_wita", "min"),
    sampai=("hour_wita", "max"),
    PM25_valid=("PM25", "count"),
    TMP_valid=("TMP", "count"),
)
display(summary)

print("Total baris:", len(df))

print("\n✓ [NB03-C03] selesai")

,jumlah_jam,mulai,sampai,PM25_valid,TMP_valid
ispu_device_id,,,,,
PH10P0001,3586,2026-04-25,2026-09-21 09:00:00,3103,2974
PH14P0002,3586,2026-04-25,2026-09-21 09:00:00,3187,3035
PH14P0003,3586,2026-04-25,2026-09-21 09:00:00,3218,3082
PH14P0004,3586,2026-04-25,2026-09-21 09:00:00,3053,2961
PH20P0001,3586,2026-04-25,2026-09-21 09:00:00,3149,3013


Total baris: 17930

✓ [NB03-C03] selesai


In [ ]:
# ============================================================
# [NB03-C04] WAKTU ORIGIN & CEK TIMELINE PER SENSOR
# ============================================================
# Baris jam 01:00 (data 01:00–01:59) lengkap pada 02:00,
# sehingga origin_time = hour_wita + 1 jam.

df["origin_time"] = df["hour_wita"] + pd.Timedelta(hours=1)
df["is_grid_origin"] = df["origin_time"].dt.hour.isin(GRID_HOURS_WITA)

# Semua fitur dan target dihitung dengan shift per sensor,
# sehingga timeline setiap sensor harus lengkap per jam (tanpa jam yang hilang).
step = df.groupby(DEVICE_COL)["hour_wita"].diff().dropna()
assert (step == pd.Timedelta(hours=1)).all(), "Timeline per jam tidak lengkap. Periksa NB00-C12."

print("Contoh:")
print(df[[DEVICE_COL, "hour_wita", "origin_time", "is_grid_origin"]].head(4).to_string(index=False))

print("\nOrigin pada jam grid :", df["is_grid_origin"].sum(), "dari", len(df), "baris")

print("\n✓ [NB03-C04] selesai")

Contoh:
ispu_device_id           hour_wita         origin_time  is_grid_origin
     PH10P0001 2026-04-25 00:00:00 2026-04-25 01:00:00           False
     PH10P0001 2026-04-25 01:00:00 2026-04-25 02:00:00            True
     PH10P0001 2026-04-25 02:00:00 2026-04-25 03:00:00           False
     PH10P0001 2026-04-25 03:00:00 2026-04-25 04:00:00           False

Origin pada jam grid : 5975 dari 17930 baris

✓ [NB03-C04] selesai


In [ ]:
# ============================================================
# [NB03-C05] FITUR LAG POLUTAN (PER SENSOR)
# ============================================================
# PM25_lag1  = rata-rata jam terakhir sebelum origin
# PM25_lag24 = rata-rata jam 24 jam sebelum origin

g = df.groupby(DEVICE_COL)

lag_features = []

for col in POLLUTANT_COLS:
    for lag in POLLUTANT_LAGS:
        name = f"{col}_lag{lag}"
        df[name] = g[col].shift(lag - 1)
        lag_features.append(name)

print("Jumlah fitur lag:", len(lag_features))
print(lag_features[:6], "...")

print("\n✓ [NB03-C05] selesai")

Jumlah fitur lag: 36
['PM25_lag1', 'PM25_lag2', 'PM25_lag3', 'PM25_lag6', 'PM25_lag12', 'PM25_lag24'] ...

✓ [NB03-C05] selesai


In [ ]:
# ============================================================
# [NB03-C06] FITUR RATA-RATA BERGERAK & PROFIL 24 JAM POLUTAN (PER SENSOR)
# ============================================================
# PM25_mean3h = rata-rata 3 jam terakhir sebelum origin
# (minimal setengah jendela harus tersedia)

rolling_features = []

for col in POLLUTANT_COLS:
    for window in POLLUTANT_ROLLING:
        name = f"{col}_mean{window}h"
        min_periods = int(np.ceil(window / 2))
        df[name] = g[col].transform(
            lambda s, w=window, m=min_periods: s.rolling(w, min_periods=m).mean()
        )
        rolling_features.append(name)

# Profil 24 jam terakhir: nilai titik grafik (rata-rata 3 jam) pada 3, 6, ..., 21 jam
# sebelum origin. Dengan fitur ini model tahu nilai "kemarin di jam yang sama"
# untuk setiap titik forecast (contoh: target +3 jam -> titik grafik 21 jam lalu).
profile_features = []

for col in POLLUTANT_COLS:
    for ago in PROFILE_HOURS_AGO:
        name = f"{col}_mean3h_ago{ago}"
        df[name] = g[f"{col}_mean3h"].shift(ago)
        profile_features.append(name)

print("Jumlah fitur rolling      :", len(rolling_features))
print(rolling_features[:3], "...")
print("Jumlah fitur profil 24 jam:", len(profile_features))
print(profile_features[:7], "...")

print("\n✓ [NB03-C06] selesai")

Jumlah fitur rolling      : 18
['PM25_mean3h', 'PM25_mean6h', 'PM25_mean24h'] ...
Jumlah fitur profil 24 jam: 42
['PM25_mean3h_ago3', 'PM25_mean3h_ago6', 'PM25_mean3h_ago9', 'PM25_mean3h_ago12', 'PM25_mean3h_ago15', 'PM25_mean3h_ago18', 'PM25_mean3h_ago21'] ...

✓ [NB03-C06] selesai


In [ ]:
# ============================================================
# [NB03-C07] FITUR CUACA (PER SENSOR)
# ============================================================
# Hanya kondisi cuaca yang sudah terjadi (sampai origin).

weather_features = []

# Cuaca jam terakhir
for col in WEATHER_COLS:
    name = f"{col}_lag1"
    df[name] = df[col]
    weather_features.append(name)

# Rata-rata 3 jam terakhir
for col in ["TMP", "HUM", "RAIN", "WS"]:
    name = f"{col}_mean3h"
    df[name] = g[col].transform(lambda s: s.rolling(3, min_periods=2).mean())
    weather_features.append(name)

# Hujan 24 jam terakhir
df["RAIN_mean24h"] = g["RAIN"].transform(lambda s: s.rolling(24, min_periods=12).mean())
weather_features.append("RAIN_mean24h")

print("Jumlah fitur cuaca:", len(weather_features))
print(weather_features)

print("\n✓ [NB03-C07] selesai")

Jumlah fitur cuaca: 11
['TMP_lag1', 'HUM_lag1', 'RAIN_lag1', 'WS_lag1', 'WD_sin_lag1', 'WD_cos_lag1', 'TMP_mean3h', 'HUM_mean3h', 'RAIN_mean3h', 'WS_mean3h', 'RAIN_mean24h']

✓ [NB03-C07] selesai


In [ ]:
# ============================================================
# [NB03-C08] FITUR WAKTU (WITA) & SENSOR
# ============================================================

origin_hour = df["origin_time"].dt.hour

df["hour_sin"] = np.sin(2 * np.pi * origin_hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * origin_hour / 24)
df["day_of_week"] = df["origin_time"].dt.dayofweek   # 0 = Senin

time_features = ["hour_sin", "hour_cos", "day_of_week"]

# One-hot sensor: model tahu data berasal dari sensor mana
sensor_dummies = pd.get_dummies(df[DEVICE_COL], prefix="sensor", dtype=int)
df = pd.concat([df, sensor_dummies], axis=1)
sensor_features = sensor_dummies.columns.tolist()

print("Fitur waktu :", time_features)
print("Fitur sensor:", sensor_features)

print("\n✓ [NB03-C08] selesai")

Fitur waktu : ['hour_sin', 'hour_cos', 'day_of_week']
Fitur sensor: ['sensor_PH10P0001', 'sensor_PH14P0002', 'sensor_PH14P0003', 'sensor_PH14P0004', 'sensor_PH20P0001']

✓ [NB03-C08] selesai


In [ ]:
# ============================================================
# [NB03-C09] TARGET: RATA-RATA 3 JAM PADA SETIAP HORIZON
# ============================================================
# PM25_block3h      = nilai titik grafik pada origin_time (aktual, rata-rata 3 jam terakhir)
# PM25_target_24h   = nilai titik grafik 24 jam setelah origin

g = df.groupby(DEVICE_COL)

block_cols = []
target_cols = []

for col in POLLUTANT_COLS:
    block_name = f"{col}_block3h"
    df[block_name] = g[col].transform(
        lambda s: s.rolling(BLOCK_HOURS, min_periods=MIN_HOURS_IN_BLOCK).mean()
    )
    block_cols.append(block_name)

    for h in ALL_HORIZONS:
        target_name = f"{col}_target_{h}h"
        df[target_name] = df.groupby(DEVICE_COL)[block_name].shift(-h)
        target_cols.append(target_name)

print("Jumlah kolom target:", len(target_cols), f"({len(POLLUTANT_COLS)} polutan × {len(ALL_HORIZONS)} horizon)")

print("\n✓ [NB03-C09] selesai")

Jumlah kolom target: 60 (6 polutan × 10 horizon)

✓ [NB03-C09] selesai


In [ ]:
# ============================================================
# [NB03-C10] VERIFIKASI FITUR & TARGET (CEK MANUAL)
# ============================================================
# Hitung ulang satu contoh secara manual dari data per jam,
# lalu bandingkan dengan kolom yang dibuat otomatis.

sample = df[df["is_grid_origin"] & df["PM25_target_24h"].notna() & df["PM25_lag24"].notna()].iloc[500]

device = sample[DEVICE_COL]
t0 = sample["origin_time"]
dev = df[df[DEVICE_COL] == device].set_index("hour_wita")

print(f"Contoh : sensor {device}, origin {t0}")

# 1. Lag: PM25_lag6 = baris jam (t0 - 6 jam)
manual_lag6 = dev.loc[t0 - pd.Timedelta(hours=6), "PM25"]
print(f"PM25_lag6        : otomatis={sample['PM25_lag6']:.4f} | manual={manual_lag6:.4f}")
assert np.isclose(sample["PM25_lag6"], manual_lag6)

# 2. Target +24 jam: rata-rata baris jam (t0+21), (t0+22), (t0+23)
target_hours = [t0 + pd.Timedelta(hours=k) for k in [21, 22, 23]]
manual_target = dev.loc[target_hours, "PM25"].mean()
print(f"PM25_target_24h  : otomatis={sample['PM25_target_24h']:.4f} | manual={manual_target:.4f}")
print(f"  titik grafik   : {t0 + pd.Timedelta(hours=24)} (rata-rata {target_hours[0].strftime('%H:%M')}–{(target_hours[-1] + pd.Timedelta(hours=1)).strftime('%H:%M')})")
assert np.isclose(sample["PM25_target_24h"], manual_target)

# 3. Profil 24 jam: PM25_mean3h_ago21 = rata-rata baris jam (t0-24), (t0-23), (t0-22)
profile_hours = [t0 - pd.Timedelta(hours=k) for k in [24, 23, 22]]
manual_profile = dev.loc[profile_hours, "PM25"].mean()
print(f"PM25_mean3h_ago21: otomatis={sample['PM25_mean3h_ago21']:.4f} | manual={manual_profile:.4f}")
assert np.isclose(sample["PM25_mean3h_ago21"], manual_profile)

# 4. Tidak ada kebocoran: jam terakhir yang dipakai fitur < origin <= jam pertama target
last_feature_hour = t0 - pd.Timedelta(hours=1)
first_target_hour = t0 + pd.Timedelta(hours=21)
assert last_feature_hour < t0 <= first_target_hour

print("\n✓ Fitur hanya memakai data sebelum origin, target berada setelah origin.")

print("\n✓ [NB03-C10] selesai")

Contoh : sensor PH10P0001, origin 2026-07-18 14:00:00
PM25_lag6        : otomatis=6.3383 | manual=6.3383
PM25_target_24h  : otomatis=5.3739 | manual=5.3739
  titik grafik   : 2026-07-19 14:00:00 (rata-rata 11:00–14:00)
PM25_mean3h_ago21: otomatis=8.5342 | manual=8.5342

✓ Fitur hanya memakai data sebelum origin, target berada setelah origin.

✓ [NB03-C10] selesai


In [ ]:
# ============================================================
# [NB03-C11] DAFTAR FITUR & RINGKASAN DATA VALID
# ============================================================

FEATURE_COLS = lag_features + rolling_features + profile_features + weather_features + time_features + sensor_features

print("=== JUMLAH FITUR ===")
print(f"Lag polutan      : {len(lag_features)}")
print(f"Rolling polutan  : {len(rolling_features)}")
print(f"Profil 24 jam    : {len(profile_features)}")
print(f"Cuaca            : {len(weather_features)}")
print(f"Waktu            : {len(time_features)}")
print(f"Sensor           : {len(sensor_features)}")
print(f"TOTAL            : {len(FEATURE_COLS)}")

features_complete = df[FEATURE_COLS].notna().all(axis=1)
print(f"\nBaris dengan semua fitur lengkap: {features_complete.sum()} dari {len(df)} "
      f"({features_complete.mean():.1%})")

# Jumlah baris valid (fitur lengkap + target ada) per polutan dan horizon
valid_counts = pd.DataFrame(
    {
        f"{h}h": [
            int((features_complete & df[f"{col}_target_{h}h"].notna()).sum())
            for col in POLLUTANT_COLS
        ]
        for h in ALL_HORIZONS
    },
    index=POLLUTANT_COLS,
)

print("\nBaris valid per target (fitur lengkap + target ada):")
print(valid_counts.to_string())

print("\n✓ [NB03-C11] selesai")

=== JUMLAH FITUR ===
Lag polutan      : 36
Rolling polutan  : 18
Profil 24 jam    : 42
Cuaca            : 11
Waktu            : 3
Sensor           : 5
TOTAL            : 115

Baris dengan semua fitur lengkap: 13527 dari 17930 (75.4%)

Baris valid per target (fitur lengkap + target ada):
         3h     6h     9h    12h    15h    18h    21h    24h    48h    72h
PM25  13448  13364  13315  13264  13205  13174  13129  13078  12682  12332
PM10  13446  13362  13313  13262  13203  13172  13127  13076  12680  12330
CO    13448  13364  13315  13264  13205  13174  13129  13078  12682  12332
O3    13448  13364  13315  13264  13205  13174  13129  13078  12682  12332
SO2   13448  13364  13315  13264  13205  13174  13129  13078  12682  12332
NO2   13448  13364  13315  13264  13205  13174  13129  13078  12682  12332

✓ [NB03-C11] selesai


In [ ]:
# ============================================================
# [NB03-C12] SPLIT BERDASARKAN WAKTU (SAMA UNTUK SEMUA SENSOR)
# ============================================================
# Batas split dihitung dari rentang waktu, bukan dari jumlah baris,
# sehingga semua sensor memakai tanggal potong yang sama.
# Catatan untuk NB04: baris train/validation yang targetnya melewati
# batas split harus dibuang (origin + horizon > batas) agar tidak bocor.

t_start = df["origin_time"].min()
t_end = df["origin_time"].max()
span = t_end - t_start

train_end = (t_start + span * TRAIN_FRAC).floor("1h")
val_end = (t_start + span * (TRAIN_FRAC + VAL_FRAC)).floor("1h")

df["split"] = np.select(
    [df["origin_time"] <= train_end, df["origin_time"] <= val_end],
    ["train", "val"],
    default="test",
)

print(f"Train      : {t_start}  →  {train_end}")
print(f"Validation : {train_end + pd.Timedelta(hours=1)}  →  {val_end}")
print(f"Test       : {val_end + pd.Timedelta(hours=1)}  →  {t_end}")

split_table = pd.crosstab(df[DEVICE_COL], df["split"])[["train", "val", "test"]]
split_table.loc["TOTAL"] = split_table.sum()
print("\nJumlah baris per sensor:")
print(split_table.to_string())

print("\n✓ [NB03-C12] selesai")

Train      : 2026-04-25 01:00:00  →  2026-08-07 14:00:00
Validation : 2026-08-07 15:00:00  →  2026-08-30 00:00:00
Test       : 2026-08-30 01:00:00  →  2026-09-21 10:00:00

Jumlah baris per sensor:
split           train   val  test
ispu_device_id                   
PH10P0001        2510   538   538
PH14P0002        2510   538   538
PH14P0003        2510   538   538
PH14P0004        2510   538   538
PH20P0001        2510   538   538
TOTAL           12550  2690  2690

✓ [NB03-C12] selesai


In [ ]:
# ============================================================
# [NB03-C13] SIMPAN DATASET MODEL & KONFIGURASI
# ============================================================

id_cols = [DEVICE_COL, "hour_wita", "origin_time", "is_grid_origin", "split"]

model_dataset = df[id_cols + block_cols + FEATURE_COLS + target_cols].copy()

float_cols = model_dataset.select_dtypes("float").columns
model_dataset[float_cols] = model_dataset[float_cols].round(4)

model_dataset.to_csv(OUTPUT_PATH, index=False)

config = {
    "db_to_wita_hours": DB_TO_WITA_HOURS,
    "start_wita": str(START_WITA),
    "pollutants": POLLUTANT_COLS,
    "horizons": HORIZONS,
    "experiment_horizons": EXPERIMENT_HORIZONS,
    "grid_hours_wita": GRID_HOURS_WITA,
    "block_hours": BLOCK_HOURS,
    "feature_cols": FEATURE_COLS,
    "block_cols": block_cols,
    "target_cols": target_cols,
    "train_end": str(train_end),
    "val_end": str(val_end),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

# Verifikasi: file bisa dibaca ulang
check = pd.read_csv(OUTPUT_PATH, parse_dates=["hour_wita", "origin_time"])
assert check.shape == model_dataset.shape
assert not check.duplicated([DEVICE_COL, "origin_time"]).any()

print("Dataset disimpan :", OUTPUT_PATH.resolve())
print("Shape            :", model_dataset.shape)
print("Config disimpan  :", CONFIG_PATH.resolve())

print("\n✓ [NB03-C13] selesai")

Dataset disimpan : D:\Kuliah\KP\forecast-ispu\data\processed\model_dataset.csv
Shape            : (17930, 186)
Config disimpan  : D:\Kuliah\KP\forecast-ispu\data\processed\model_config.json

✓ [NB03-C13] selesai
